# AI Task 1: Demand Forecasting (Preparation Order Generation)

**Goal:** Predict which SKUs and what quantities will be needed for delivery the next day.

**Approach:** Time series forecasting with LightGBM  
- Per-product features: lags, rolling means, day-of-week, etc.  
- Target: `y(p, t+1)` = demand for product `p` on day `t+1`  
- Time-based train/test split (80/20)  
- Metrics: RMSE, MAE, MAPE, MASE, service-level accuracy

## 1. Install & Import Dependencies

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

try:
    import lightgbm as lgb
    print("LightGBM version:", lgb.__version__)
except ImportError:
    print("Installing lightgbm...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'lightgbm'])
    import lightgbm as lgb
    print("LightGBM installed:", lgb.__version__)

import matplotlib.pyplot as plt

print("All imports ready!")

LightGBM version: 4.6.0
All imports ready!


## 2. Load Data & Encode Categorical Features

In [2]:
# Load the grouped data
df = pd.read_csv('products_for_ts_grouped.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['date'].min()} → {df['date'].max()}")
print(f"Unique products: {df['id_produit'].nunique()}")
print(f"Unique categories: {df['categorie'].nunique()}")
print(f"\nCategories: {df['categorie'].unique()}")

# --- Date parsing ---
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

# --- Encode Is_Gerbable (True/False → 1/0) ---
df['Is_Gerbable'] = df['Is_Gerbable'].astype(str).str.strip().str.lower().map({'true': 1, 'false': 0})

# --- One-hot encode categorie ---
df = pd.get_dummies(df, columns=['categorie'], prefix='cat', dtype=int)

print("\nEncoded columns:")
print(list(df.columns))

print("\nFirst 5 rows:")
df.head()

Shape: (80651, 8)
Columns: ['id_produit', 'date', 'categorie', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'quantite_demande']

Date range: 2024-01-02 → 2026-01-08
Unique products: 1124
Unique categories: 29

Categories: ['MOULURE' 'MONO ACCESSOIRES' 'MINI DISJONCTEUR' 'CHROME'
 'TABLEAUX DISTRIBUTION' 'DPN' 'DISPINA' 'DISPINA METALIC'
 'EVOLUTION PLUS' 'EVOLUTION' 'LARISSA' 'SPOT' 'LAMPE LED MONO' 'MODULE'
 'SLIM LED' 'OBERON' 'OCTANS' 'STYLE' 'LEON' 'ICON' 'ORION' 'Tous'
 'TUBE LED' 'REGLETTE' 'ACCESSOIRES' 'LEON NOIR' 'PROJECTEUR NOIR' 'MOON'
 'DISJONCTEUR']

Encoded columns:
['id_produit', 'date', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'quantite_demande', 'cat_ACCESSOIRES', 'cat_CHROME', 'cat_DISJONCTEUR', 'cat_DISPINA', 'cat_DISPINA METALIC', 'cat_DPN', 'cat_EVOLUTION', 'cat_EVOLUTION PLUS', 'cat_ICON', 'cat_LAMPE LED MONO', 'cat_LARISSA', 'cat_LEON', 'cat_LEON NOIR', 'cat_MINI DISJONCTEUR', 'cat_MODULE', 'cat_MONO A

,id_produit,date,colisage fardeau,colisage palette,volume pcs (m3),Is_Gerbable,quantite_demande,cat_ACCESSOIRES,cat_CHROME,cat_DISJONCTEUR,...,cat_OCTANS,cat_ORION,cat_PROJECTEUR NOIR,cat_REGLETTE,cat_SLIM LED,cat_SPOT,cat_STYLE,cat_TABLEAUX DISTRIBUTION,cat_TUBE LED,cat_Tous
0,31334,2024-01-07,32,1600,0.0002,1,544,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,31334,2024-01-09,32,1600,0.0002,1,64,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,31334,2024-01-10,32,1600,0.0002,1,32,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,31334,2024-01-11,32,1600,0.0002,1,64,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,31334,2024-01-15,32,1600,0.0002,1,192,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. Fill Missing Days & Build Continuous Time Series

Products don't have entries for every day (no demand = missing row). We need to fill gaps with 0 so lag/rolling features work correctly.

In [3]:
# Create a complete daily date range
date_min = df['date'].min()
date_max = df['date'].max()
all_dates = pd.date_range(start=date_min, end=date_max, freq='D')
all_products = df['id_produit'].unique()

print(f"Date range: {date_min.date()} → {date_max.date()} ({len(all_dates)} days)")
print(f"Products: {len(all_products)}")
print(f"Expected rows: {len(all_dates) * len(all_products):,}")

# Create multi-index of all (product, date) combinations
idx = pd.MultiIndex.from_product([all_products, all_dates], names=['id_produit', 'date'])
df_full = df.set_index(['id_produit', 'date']).reindex(idx).reset_index()

# Fill demand with 0 for missing days
df_full['quantite_demande'] = df_full['quantite_demande'].fillna(0)

# Forward-fill static product features (they don't change per product)
static_cols = [c for c in df_full.columns if c not in ['id_produit', 'date', 'quantite_demande']]
df_full[static_cols] = df_full.groupby('id_produit')[static_cols].ffill().bfill()

# Sort
df_full = df_full.sort_values(['id_produit', 'date']).reset_index(drop=True)

print(f"\nFull dataset shape: {df_full.shape}")
print(f"Rows with 0 demand: {(df_full['quantite_demande'] == 0).sum():,}")
print(f"Rows with > 0 demand: {(df_full['quantite_demande'] > 0).sum():,}")
df_full.head(10)

Date range: 2024-01-02 → 2026-01-08 (738 days)
Products: 1124
Expected rows: 829,512

Full dataset shape: (829512, 36)
Rows with 0 demand: 748,861
Rows with > 0 demand: 80,651


,id_produit,date,colisage fardeau,colisage palette,volume pcs (m3),Is_Gerbable,quantite_demande,cat_ACCESSOIRES,cat_CHROME,cat_DISJONCTEUR,...,cat_OCTANS,cat_ORION,cat_PROJECTEUR NOIR,cat_REGLETTE,cat_SLIM LED,cat_SPOT,cat_STYLE,cat_TABLEAUX DISTRIBUTION,cat_TUBE LED,cat_Tous
0,31334,2024-01-02,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,31334,2024-01-03,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,31334,2024-01-04,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,31334,2024-01-05,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31334,2024-01-06,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,31334,2024-01-07,32.0,1600.0,0.0002,1.0,544.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,31334,2024-01-08,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,31334,2024-01-09,32.0,1600.0,0.0002,1.0,64.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,31334,2024-01-10,32.0,1600.0,0.0002,1.0,32.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,31334,2024-01-11,32.0,1600.0,0.0002,1.0,64.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Feature Engineering

For each product `p` and day `t`, build input features:
- **Calendar**: day_of_week, month, day_of_month, is_weekend, is_monday, week_of_year
- **Lag features**: demand at t-1, t-2, t-3, t-7, t-14, t-28
- **Rolling means**: 3-day, 7-day, 14-day, 28-day rolling average
- **Rolling std**: 7-day, 14-day rolling std (volatility)
- **Days since last order**: how many days since the last non-zero demand
- **Target**: demand at t+1 (next day)

In [4]:
# --- Calendar features ---
df_full['day_of_week'] = df_full['date'].dt.dayofweek        # 0=Monday ... 6=Sunday
df_full['month'] = df_full['date'].dt.month
df_full['day_of_month'] = df_full['date'].dt.day
df_full['week_of_year'] = df_full['date'].dt.isocalendar().week.astype(int)
df_full['is_weekend'] = (df_full['day_of_week'] >= 5).astype(int)
df_full['is_monday'] = (df_full['day_of_week'] == 0).astype(int)

# --- Lag features (per product) ---
for lag in [1, 2, 3, 7, 14, 28]:
    df_full[f'lag_{lag}'] = df_full.groupby('id_produit')['quantite_demande'].shift(lag)

# --- Rolling mean features (per product) ---
for window in [3, 7, 14, 28]:
    df_full[f'rolling_mean_{window}'] = (
        df_full.groupby('id_produit')['quantite_demande']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

# --- Rolling std features (per product) ---
for window in [7, 14]:
    df_full[f'rolling_std_{window}'] = (
        df_full.groupby('id_produit')['quantite_demande']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    )

# --- Days since last non-zero demand (per product) ---
def days_since_last_order(group):
    last_order_date = pd.NaT
    result = []
    for _, row in group.iterrows():
        if last_order_date is pd.NaT:
            result.append(np.nan)
        else:
            result.append((row['date'] - last_order_date).days)
        if row['quantite_demande'] > 0:
            last_order_date = row['date']
    return result

df_full['days_since_last_order'] = df_full.groupby('id_produit', group_keys=False).apply(
    lambda g: pd.Series(days_since_last_order(g), index=g.index)
)

# --- Target: next day demand ---
df_full['target'] = df_full.groupby('id_produit')['quantite_demande'].shift(-1)

print(f"Features built! Shape: {df_full.shape}")
print(f"\nAll columns ({len(df_full.columns)}):")
print(list(df_full.columns))

# Show sample
df_full[df_full['id_produit'] == df_full['id_produit'].iloc[0]].head(10)

Features built! Shape: (829512, 56)

All columns (56):
['id_produit', 'date', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'quantite_demande', 'cat_ACCESSOIRES', 'cat_CHROME', 'cat_DISJONCTEUR', 'cat_DISPINA', 'cat_DISPINA METALIC', 'cat_DPN', 'cat_EVOLUTION', 'cat_EVOLUTION PLUS', 'cat_ICON', 'cat_LAMPE LED MONO', 'cat_LARISSA', 'cat_LEON', 'cat_LEON NOIR', 'cat_MINI DISJONCTEUR', 'cat_MODULE', 'cat_MONO ACCESSOIRES', 'cat_MOON', 'cat_MOULURE', 'cat_OBERON', 'cat_OCTANS', 'cat_ORION', 'cat_PROJECTEUR NOIR', 'cat_REGLETTE', 'cat_SLIM LED', 'cat_SPOT', 'cat_STYLE', 'cat_TABLEAUX DISTRIBUTION', 'cat_TUBE LED', 'cat_Tous', 'day_of_week', 'month', 'day_of_month', 'week_of_year', 'is_weekend', 'is_monday', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_3', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_14', 'days_since_last_order', 'target']


,id_produit,date,colisage fardeau,colisage palette,volume pcs (m3),Is_Gerbable,quantite_demande,cat_ACCESSOIRES,cat_CHROME,cat_DISJONCTEUR,...,lag_14,lag_28,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_std_7,rolling_std_14,days_since_last_order,target
0,31334,2024-01-02,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,31334,2024-01-03,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,0.0
2,31334,2024-01-04,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.0
3,31334,2024-01-05,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.0
4,31334,2024-01-06,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,544.0
5,31334,2024-01-07,32.0,1600.0,0.0002,1.0,544.0,0.0,0.0,0.0,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.0
6,31334,2024-01-08,32.0,1600.0,0.0002,1.0,0.0,0.0,0.0,0.0,...,NaN,NaN,181.333333,90.666667,90.666667,90.666667,222.087070,222.087070,1.0,64.0
7,31334,2024-01-09,32.0,1600.0,0.0002,1.0,64.0,0.0,0.0,0.0,...,NaN,NaN,181.333333,77.714286,77.714286,77.714286,205.612673,205.612673,2.0,32.0
8,31334,2024-01-10,32.0,1600.0,0.0002,1.0,32.0,0.0,0.0,0.0,...,NaN,NaN,202.666667,86.857143,76.000000,76.000000,202.987215,190.422088,1.0,64.0
9,31334,2024-01-11,32.0,1600.0,0.0002,1.0,64.0,0.0,0.0,0.0,...,NaN,NaN,32.000000,91.428571,71.111111,71.111111,201.056258,178.726358,1.0,0.0


## 5. Prepare Train / Test Split (Time-Based)

- **Train**: oldest 80% of dates  
- **Test**: most recent 20% of dates  
- Drop rows with NaN (from lags / target at boundaries)

In [5]:
# Drop rows where target or lag features are NaN
df_model = df_full.dropna(subset=['target', 'lag_1']).copy()

print(f"Rows after dropping NaN: {len(df_model):,}")

# Time-based split: 80% train, 20% test
unique_dates = sorted(df_model['date'].unique())
split_idx = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_idx]

train = df_model[df_model['date'] < split_date].copy()
test = df_model[df_model['date'] >= split_date].copy()

print(f"\nSplit date: {split_date.date() if hasattr(split_date, 'date') else split_date}")
print(f"Train: {train['date'].min().date()} → {train['date'].max().date()} ({len(train):,} rows)")
print(f"Test:  {test['date'].min().date()} → {test['date'].max().date()} ({len(test):,} rows)")
print(f"\nTrain products: {train['id_produit'].nunique()}")
print(f"Test products: {test['id_produit'].nunique()}")

# Define feature columns (exclude id_produit, date, target, quantite_demande)
exclude_cols = ['id_produit', 'date', 'target', 'quantite_demande']
feature_cols = [c for c in df_model.columns if c not in exclude_cols]

print(f"\nFeature columns ({len(feature_cols)}):")
print(feature_cols)

X_train = train[feature_cols]
y_train = train['target']
X_test = test[feature_cols]
y_test = test['target']

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

Rows after dropping NaN: 827,264

Split date: 2025-08-13
Train: 2024-01-03 → 2025-08-12 (660,912 rows)
Test:  2025-08-13 → 2026-01-07 (166,352 rows)

Train products: 1124
Test products: 1124

Feature columns (52):
['colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'cat_ACCESSOIRES', 'cat_CHROME', 'cat_DISJONCTEUR', 'cat_DISPINA', 'cat_DISPINA METALIC', 'cat_DPN', 'cat_EVOLUTION', 'cat_EVOLUTION PLUS', 'cat_ICON', 'cat_LAMPE LED MONO', 'cat_LARISSA', 'cat_LEON', 'cat_LEON NOIR', 'cat_MINI DISJONCTEUR', 'cat_MODULE', 'cat_MONO ACCESSOIRES', 'cat_MOON', 'cat_MOULURE', 'cat_OBERON', 'cat_OCTANS', 'cat_ORION', 'cat_PROJECTEUR NOIR', 'cat_REGLETTE', 'cat_SLIM LED', 'cat_SPOT', 'cat_STYLE', 'cat_TABLEAUX DISTRIBUTION', 'cat_TUBE LED', 'cat_Tous', 'day_of_week', 'month', 'day_of_month', 'week_of_year', 'is_weekend', 'is_monday', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_3', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_

## 6. Train LightGBM Model

LightGBM is ideal for tabular time series forecasting:
- Handles mixed feature types well
- Captures non-linear patterns
- Fast training, good with many products at once

In [7]:
# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# LightGBM parameters tuned for demand forecasting
params = {
    'objective': 'regression',
    'metric': ['rmse', 'mae'],
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'n_jobs': -1,
    'verbose': -1,
    'seed': 42
}

# Train with early stopping
print("Training LightGBM model...")
callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(period=100)
]

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'test'],
    callbacks=callbacks
)

print(f"\nBest iteration: {model.best_iteration}")
print(f"Best test RMSE: {model.best_score['test']['rmse']:.4f}")
print(f"Best test MAE: {model.best_score['test'].get('mae', model.best_score['test'].get('l1', 'N/A'))}")

Training LightGBM model...
Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 326.648	train's l1: 55.3725	test's rmse: 400.917	test's l1: 64.4321
Early stopping, best iteration is:
[62]	train's rmse: 344.824	train's l1: 58.9872	test's rmse: 400.303	test's l1: 65.7413

Best iteration: 62
Best test RMSE: 400.3031
Best test MAE: 65.74125225566036


## 7. Feature Importance

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("Top 15 features by importance (gain):\n")
print(importance.head(15).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top_n = importance.head(20)
ax.barh(top_n['feature'][::-1], top_n['importance'][::-1])
ax.set_xlabel('Importance (Gain)')
ax.set_title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()

## 8. Evaluation Metrics

**Overall & per-SKU metrics:**
- RMSE (Root Mean Squared Error)
- MAE (Mean Absolute Error)  
- MAPE (Mean Absolute Percentage Error) — only on non-zero actuals
- MASE (Mean Absolute Scaled Error) — relative to naive forecast
- Service-level: % of days forecast is within ±X units of actual

In [ ]:
# Predict on test set
y_pred = model.predict(X_test, num_iteration=model.best_iteration)

# Clip negative predictions to 0 (demand can't be negative)
y_pred = np.maximum(y_pred, 0)

# --- Overall Metrics ---
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

# MAPE (only where actual > 0)
mask_nonzero = y_test > 0
if mask_nonzero.sum() > 0:
    mape = np.mean(np.abs((y_test[mask_nonzero] - y_pred[mask_nonzero]) / y_test[mask_nonzero])) * 100
else:
    mape = np.nan

# MASE: compared to naive forecast (predict lag_1)
naive_errors = np.abs(y_test - test['lag_1'].values)
model_errors = np.abs(y_test - y_pred)
mase = np.mean(model_errors) / np.mean(naive_errors) if np.mean(naive_errors) > 0 else np.nan

# Service-level: % of days within ±10 units and ±20 units
within_10 = np.mean(np.abs(y_test - y_pred) <= 10) * 100
within_20 = np.mean(np.abs(y_test - y_pred) <= 20) * 100
within_50 = np.mean(np.abs(y_test - y_pred) <= 50) * 100

print("=" * 60)
print("OVERALL TEST METRICS")
print("=" * 60)
print(f"RMSE:                     {rmse:.4f}")
print(f"MAE:                      {mae:.4f}")
print(f"MAPE (non-zero only):     {mape:.2f}%")
print(f"MASE (vs naive lag-1):    {mase:.4f}")
print(f"  → MASE < 1 means model beats naive")
print(f"\nService-Level Accuracy:")
print(f"  Within ±10 units:       {within_10:.1f}%")
print(f"  Within ±20 units:       {within_20:.1f}%")
print(f"  Within ±50 units:       {within_50:.1f}%")

## 9. Per-SKU Metrics